# 15.01 · Reasoning models & test-time compute

**Chapter:** Advanced & Frontier Topics · **Milestone:** M7

## Goals
- [ ] Explain what makes a "reasoning model" (long chain of thought trained with RL on verifiable rewards — 12.05) and how it differs from prompting CoT
- [ ] Control **thinking budgets** across providers (`reasoning_effort`, thinking budgets, `/think`) and implement **budget forcing** yourself
- [ ] Measure the two test-time scaling axes: **sequential** (think longer) and **parallel** (sample N, vote / verify)
- [ ] Plot accuracy vs tokens and decide when reasoning is worth its cost and latency

## Resources
- [DeepSeek-R1 paper](https://arxiv.org/abs/2501.12948) — free
- [s1: Simple test-time scaling (budget forcing)](https://arxiv.org/abs/2501.19393) · [Self-consistency](https://arxiv.org/abs/2203.11171) — free
- [Scaling test-time compute (Snell et al.)](https://arxiv.org/abs/2408.03314) — free
- Provider docs: [OpenAI reasoning](https://platform.openai.com/docs/guides/reasoning) · [Anthropic extended thinking](https://docs.claude.com/en/docs/build-with-claude/extended-thinking) ·
  [Gemini thinking](https://ai.google.dev/gemini-api/docs/thinking) — free

A reasoning model writes a long scratchpad (the "thinking") before its answer. It was trained to do that with **RL against verifiable
rewards** (math answers, unit tests: exactly the GRPO loop of 12.05), so it learned to check its work, backtrack and try alternatives.
More thinking tokens usually mean higher accuracy on hard multi-step problems — and more cost and latency. Your job is to pick the budget.

| Provider | Knob |
|---|---|
| OpenAI (o-series, GPT-5), Groq gpt-oss | `reasoning_effort`: minimal / low / medium / high |
| Anthropic Claude | `thinking={"type": "enabled", "budget_tokens": N}` (or adaptive effort) |
| Gemini 2.5+ | `thinking_budget` (tokens; 0 disables on Flash) / `reasoning_effort` on the OpenAI endpoint |
| Qwen3 / open models | `/think` `/no_think`, `enable_thinking`, or cap the thinking yourself (budget forcing, below) |

In [ ]:
import random
import re
import time
from collections import Counter

import httpx
import matplotlib.pyplot as plt
import numpy as np

from aieng.llm import available, client

## 1. A task set with exact answers
Two kinds of problems small models get wrong "at a glance" but can solve step by step: multi-step arithmetic and letter counting
(tokenizers hide letters — 04.01 — so the model must spell the word out).

In [ ]:
def arithmetic(rng):
    a, b, c, d, e, f = (rng.randint(11, 99) for _ in range(6))
    expr = rng.choice([f"({a} * {b}) - ({c} + {d}) + {e} * {f}", f"{a} * {b} - {c} * {d} + {e} - {f}", f"({a} + {b}) * ({c} - {d}) + {e}"])
    return f"What is {expr}?", eval(expr)


def letters(rng):
    letter = rng.choice("rst")
    word = "".join(rng.choice(letter * 3 + "aeiouln") for _ in range(rng.randint(14, 20)))
    return f"How many times does the letter '{letter}' appear in the string '{word}'?", word.count(letter)


rng = random.Random(0)
PROBLEMS = [arithmetic(rng) for _ in range(12)] + [letters(rng) for _ in range(12)]
for q, a in PROBLEMS[::6]:
    print(f"{a:>6}  {q}")

## 2. Budget forcing with a local reasoning model
Qwen3 thinks inside `<think>…</think>`. Budget forcing (s1 paper): let it think for at most *B* tokens, then **force** the end of thinking
and the start of the answer. B = 0 is "no thinking". We use Ollama's raw mode so we control the exact prompt.

In [ ]:
OLLAMA = "http://localhost:11434/api/generate"
LOCAL = "qwen3:1.7b"


def generate(prompt, max_tokens, temperature=0.0, stop=None, seed=0):
    r = httpx.post(OLLAMA, timeout=600, json={"model": LOCAL, "prompt": prompt, "raw": True, "think": False, "stream": False,
                   "options": {"num_predict": max_tokens, "temperature": temperature, "seed": seed, **({"stop": stop} if stop else {})}}).json()
    return r["response"], r["eval_count"]


def solve(question, budget, temperature=0.0, seed=0):
    """Returns (answer int or None, tokens generated)."""
    prompt = f"<|im_start|>user\n{question} Put the final answer in \\boxed{{}}.<|im_end|>\n<|im_start|>assistant\n<think>\n"
    thinking, used = generate(prompt, budget, temperature, ["</think>"], seed) if budget else ("", 0)
    answer, n = generate(prompt + thinking + "\n</think>\n\nThe final answer is $\\boxed{", 12, temperature, ["}"], seed)
    m = re.search(r"-?\d+", answer.replace(",", ""))
    return (int(m.group()) if m else None), used + n


def accuracy(budget, problems=PROBLEMS):
    start, correct, tokens = time.perf_counter(), [], []
    for q, a in problems:
        pred, used = solve(q, budget)
        correct.append(pred == a); tokens.append(used)
    return np.mean(correct), np.mean(tokens), time.perf_counter() - start


sequential = {}
for budget in (0, 128, 512, 1536):
    acc, tok, secs = accuracy(budget)
    sequential[budget] = (acc, tok)
    print(f"budget {budget:5}: accuracy {acc:4.0%} | {tok:6.0f} tokens/problem | {secs:4.0f}s")

## 3. Parallel scaling: self-consistency
Instead of one long chain, sample *N* answers at temperature > 0 and take the **majority vote**. Works best when wrong answers are scattered
and right answers agree. With a **verifier** (unit tests, a checker, a reward model) you can do even better: best-of-N picks a verified answer.

In [ ]:
def self_consistency(budget, n, problems=PROBLEMS):
    correct, tokens = [], []
    for q, a in problems:
        votes, used = [], 0
        for seed in range(n):
            pred, t = solve(q, budget, temperature=0.8, seed=seed)
            votes.append(pred); used += t
        majority = Counter(v for v in votes if v is not None).most_common(1)
        correct.append(bool(majority) and majority[0][0] == a); tokens.append(used)
    return np.mean(correct), np.mean(tokens)


parallel = {}
for budget, n in [(0, 5), (0, 15), (128, 5)]:
    parallel[(budget, n)] = self_consistency(budget, n)
    print(f"budget {budget:4} × {n:2} samples, majority vote: accuracy {parallel[(budget, n)][0]:4.0%} | {parallel[(budget, n)][1]:6.0f} tokens/problem")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
acc, tok = zip(*sequential.values())
ax.plot(tok, acc, "o-", label="sequential: think longer (budget forcing)")
for (bud, n), (a, t) in parallel.items():
    ax.scatter(t, a, marker="s", s=60, label=f"parallel: budget {bud} × {n} votes")
for budget, (a, t) in sequential.items():
    ax.annotate(f"B={budget}", (t, a), textcoords="offset points", xytext=(5, -12), fontsize=8)
ax.set_xscale("symlog", linthresh=20); ax.set_xlabel("tokens generated per problem (log)"); ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.set_title(f"{LOCAL}: test-time compute vs accuracy"); plt.show()

What to look for (on this set, qwen3:1.7b typically goes from ~5% with no thinking to ~70% at 1.5k thinking tokens):
- **Sequential scaling works** and hasn't flattened yet at 1.5k tokens: this model needs room to spell out letters and carry digits.
- **Voting can't fix a model that's consistently wrong**: 15 no-think votes stay near zero — the errors aren't random, they're systematic.
- **At equal tokens, thinking longer beats voting here** (budget 512 vs 128 × 5). Voting shines when single answers are *right more often than not*
  but noisy; reasoning shines when the problem needs intermediate steps.
Per-type breakdown is an exercise — letter counting and arithmetic often respond differently.

## 4. API reasoning models: the same trade-off, one knob
Groq's `gpt-oss-20b` (free tier) exposes `reasoning_effort`. Reasoning tokens are billed as output tokens even though you don't see them.

In [ ]:
if "groq" in available():
    gc, _ = client("groq")
    subset = PROBLEMS[::2]
    for effort in ("low", "medium", "high"):
        correct, tokens, start = [], [], time.perf_counter()
        for q, a in subset:
            for attempt in range(3):
                try:
                    r = gc.chat.completions.create(model="openai/gpt-oss-20b", reasoning_effort=effort, max_tokens=6000,
                                                   messages=[{"role": "user", "content": q + " Answer with only the final number."}])
                    break
                except Exception as e:  # free-tier tokens/minute limits
                    time.sleep(20)
            m = re.findall(r"-?\d+", (r.choices[0].message.content or "").replace(",", ""))
            correct.append(bool(m) and int(m[-1]) == a); tokens.append(r.usage.completion_tokens)
            time.sleep(2)
        print(f"gpt-oss-20b effort={effort:6}: accuracy {np.mean(correct):4.0%} | {np.mean(tokens):5.0f} output tokens | "
              f"{(time.perf_counter() - start) / len(subset):.1f}s/problem")
else:
    print("GROQ_API_KEY not set — skipping")

A stronger model needs far fewer tokens for the same problems. More effort isn't monotonic on a 12-problem sample (one or two problems are
noise): pick the **lowest effort that meets your accuracy target** on a large enough eval, because cost and latency grow with effort.

## 5. When to use reasoning
- **Yes**: multi-step math/logic, code with tricky constraints, planning, agentic tasks that need to recover from errors, hard extraction with rules.
- **No**: lookup, classification, summarisation, chit-chat, anything latency-critical — reasoning adds seconds and multiplies output tokens.
- **Route** (11.02): a cheap no-think call first; escalate to high effort only when a check fails (13.06 cascade).
- Don't over-prompt reasoning models: skip "think step by step" and long few-shot CoT; give the goal, constraints and output format.
- Thinking text is not a faithful explanation of the computation (and providers often hide or summarise it). Evaluate the **answers**.
- Context: thinking tokens fill the context window and count toward `max_tokens` — set limits with headroom (14.03).

## Exercises
1. Plot accuracy vs thinking budget on a hard task set **of your own** (e.g. 20 problems from your domain with checkable answers). Where does it flatten?
2. Break §2–3 down by problem type (arithmetic vs letters). Which benefits from sequential thinking, which from voting?
3. Implement the s1 "Wait" trick: when the model tries to stop thinking early, append "Wait" and let it continue. Does forcing *more* thinking help?
4. Best-of-N with a verifier: for arithmetic, have the model also output a Python expression; run it (07/09 sandbox rules) and pick the answer
   that verifies. Compare with majority voting at equal token cost.

In [ ]:
# your code here